Instalação do Pydantic

In [ ]:
!pip install pydantic
!pip install pydantic[email]

Importações das bibliotecas necessárias

In [ ]:
import enum
import hashlib
import re
from typing import Any

from pydantic import (
    BaseModel,
    EmailStr,
    Field,
    field_validator,
    model_validator,
    SecretStr,
    ValidationError,
)


Criação de alguns Regex para validar nomes e senhas

In [ ]:
VALID_PASSWORD_REGEX = re.compile(r"^(?=.*[a-z])(?=.*[A-Z])(?=.*\d).{8,}$")
VALID_NAME_REGEX = re.compile(r"^[a-zA-Z]{2,}$")

Criação da Classe Role (Função/Cargo)

In [ ]:
class Role(enum.IntFlag): # Define 4 possíveis funções com seus respectivos valores
    Author = 1
    Editor = 2
    Admin = 4
    SuperAdmin = 8

Criação da Classe User (Usuário)

In [ ]:
class User(BaseModel): # Cria um modelo de dados, usando o BaseModel do Pydantic que valida e estrutura os dados automaticamente
    name: str = Field(examples=["Arjan"])
    email: EmailStr = Field( # Valida se o valor é um email válido
        examples=["user@arjancodes.com"],
        description="The email address of the user",
        frozen=True, # Congela o campo, isto é depois de criado o email, ele não pode ser alterado
    )
    password: SecretStr = Field( # Esconde o valor do campo
        examples=["Password123"], description="The password of the user"
    )
    role: Role = Field(
        default=None, description="The role of the user", examples=[1, 2, 4, 8]
    )

    @field_validator("name") # Usado para adicionar uma validação personalizada para um objeto Pydantic
    @classmethod
    def validate_name(cls, v: str) -> str: # v = value
        if not VALID_NAME_REGEX.match(v): # Verifica se o Regex (definido anteriormente) corresponde ao valor obtido
            raise ValueError(
                "Name is invalid, must contain only letters and be at least 2 characters long"
            )
        return v
    # Field_validator valida campos específicos em um modelo
    @field_validator("role", mode="before") # O mode pode ser "before" - valor ainda não foi definido no objeto - ou "after" - significa que a instância já foi criada e não são classmethod e sim self
    @classmethod
    def validate_role(cls, v: int | str | Role) -> Role:
        op = {int: lambda x: Role(x), str: lambda x: Role[x], Role: lambda x: x}
        try:
            return op[type(v)](v)
        except (KeyError, ValueError):
            raise ValueError(
                f"Role is invalid, please use one of the following: {', '.join([x.name for x in Role])}"
            )

    @model_validator(mode="before") # Permite validar todos os dados da instância de uma vez
    @classmethod
    def validate_user(cls, v: dict[str, Any]) -> dict[str, Any]:
        if "name" not in v or "password" not in v: # Verifica se nome ou senha estão ausentes nos dados recebidos
            raise ValueError("Name and password are required")
        if v["name"].casefold() in v["password"].casefold(): # Verifica se a senha contém o nome do usuário
            raise ValueError("Password cannot contain name")
        if not VALID_PASSWORD_REGEX.match(v["password"]): # Verifica se a senha bate com a regex de segurança definida previamente
            raise ValueError(
                "Password is invalid, must contain 8 characters, 1 uppercase, 1 lowercase, 1 number"
            )
        v["password"] = hashlib.sha256(v["password"].encode()).hexdigest() # Criptografa a senha
        return v

Criação da função para validação do usuário

In [ ]:
def validate(data: dict[str, Any]) -> None:
    try:
        user = User.model_validate(data) # Permite validar se os dados estão de acordo com a estrutura dos dados do usuário (definidos anteriormente)
        print(user)
    except ValidationError as e:
        print("User is invalid:") # Ocorre caso os dados do usuário são inválidos
        print(e) # Imprimi o erro que ocorreu

Criação da Função Principal (executa a validação dos dados de exemplo)

In [ ]:
def main() -> None:
    test_data = dict(
        good_data={ # Dados corretos com a estrutura dos dados
            "name": "Arjan",
            "email": "example@arjancodes.com",
            "password": "Password123",
            "role": "Admin",
        },
        bad_role={ # Dado de role incorreto, role que não existe nas opções de role
            "name": "Arjan",
            "email": "example@arjancodes.com",
            "password": "Password123",
            "role": "Programmer",
        },
        bad_data={ # Dados incompatíveis com a estrutura de dados
            "name": "Arjan",
            "email": "bad email",
            "password": "bad password",
        },
        bad_name={ # Dado de nome incompatível com a estrutura dos dados
            "name": "Arjan<-_->",
            "email": "example@arjancodes.com",
            "password": "Password123",
        },
        duplicate={ # Senha contendo nome
            "name": "Arjan",
            "email": "example@arjancodes.com",
            "password": "Arjan123",
        },
        missing_data={ # Dados faltantes
            "email": "<bad data>",
            "password": "<bad data>",
        },
    )

    for example_name, data in test_data.items(): # Verificação de todos os exemplos de dados acima
        print(example_name)
        validate(data)
        print()


Execução da função Principal



In [ ]:
if __name__ == "__main__":
    main()

good_data
name='Arjan' email='example@arjancodes.com' password=SecretStr('**********') role=<Role.Admin: 4>

bad_role
User is invalid:
1 validation error for User
role
  Value error, Role is invalid, please use one of the following: Author, Editor, Admin, SuperAdmin [type=value_error, input_value='Programmer', input_type=str]
    For further information visit https://errors.pydantic.dev/2.12/v/value_error

bad_data
User is invalid:
1 validation error for User
  Value error, Password is invalid, must contain 8 characters, 1 uppercase, 1 lowercase, 1 number [type=value_error, input_value={'name': 'Arjan', 'email'...ssword': 'bad password'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/value_error

bad_name
User is invalid:
1 validation error for User
name
  Value error, Name is invalid, must contain only letters and be at least 2 characters long [type=value_error, input_value='Arjan<-_->', input_type=str]
    For further information visit https://